In [ ]:
# STEP 0 - run FIRST in a fresh kernel. Stops early (with advice) if the GPU is not healthy.
import os, subprocess, time
import torch
def gpu_health():
    if not torch.cuda.is_available(): raise SystemExit("CUDA is not available in this kernel. Select the 'Python (GPU)' kernel.")
    try:
        a = torch.randn(2048, 2048, device="cuda"); b = (a @ a).sum(); torch.cuda.synchronize()
        x = torch.randn(64, 3, 224, 224, device="cuda"); y = torch.nn.functional.conv2d(x, torch.randn(8, 3, 3, 3, device="cuda")); torch.cuda.synchronize()
        free, total = torch.cuda.mem_get_info(); print(f"GPU OK: {torch.cuda.get_device_name(0)} | free {free/1e9:.1f} of {total/1e9:.1f} GB")
        del a, b, x, y; torch.cuda.empty_cache(); return True
    except Exception as e:
        print("GPU TEST FAILED:", type(e).__name__, str(e)[:200])
        print("1) Kernel > Restart Kernel (a CUDA 'illegal memory access' is sticky until you restart).")
        print("2) Close every other notebook that uses the GPU and check 'nvidia-smi'.")
        print("3) If it fails again in a fresh kernel: reboot, update the NVIDIA driver, reset any GPU overclock/undervolt tool to stock.")
        return False
try: print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,temperature.gpu", "--format=csv"], capture_output=True, text=True, timeout=20).stdout)
except Exception: pass
assert gpu_health(), "GPU not healthy: follow the advice printed above"
dev = "cuda"

In [ ]:
# STEP 1 - setup
import gc, re, json, math, shutil, datetime
import numpy as np, pandas as pd, timm, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score
from tqdm import tqdm

OUT = r"G:\runs"; DG = rf"{OUT}\dg"; XD = rf"{OUT}\part34"; VW = rf"{OUT}\vinbig"
NW = rf"{OUT}\c_source"; RS = rf"{OUT}\fair_ens"                       # NW: new checkpoints | RS: every table, figure and prediction of this notebook
for p_ in (NW, RS): os.makedirs(p_, exist_ok=True)
E_OLD, EPOCHS, SEEDS, L, BS, LR = 7, 7, [0, 1, 2], 6, 64, 1e-4         # E_OLD: epochs of the models you already trained
LABELS6 = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
FIND5 = ["Cardiomegaly", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]; IDX5 = [LABELS6.index(l) for l in FIND5]
ARCHS = ["DenseNet", "ConvNeXt"]; TIMM = {"DenseNet": "densenet121", "ConvNeXt": "convnext_tiny.fb_in1k"}

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"], "mathtext.fontset": "stix", "font.size": 8,
    "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(rf"{RS}\{name}.pdf", bbox_inches="tight"); fig.savefig(rf"{RS}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()

MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
def prep(A, idx):
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - MEAN) / STD
def gpu_batch(A, idx, flip=False):                                    # identical augmentation to all your earlier runs: random horizontal flip only
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device=dev) < 0.5; x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD
@torch.inference_mode()
def predict_logits_idx(m, A, idx, bs=128):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast(dev, enabled=(dev == "cuda")): o = m(prep(A, idx[i:i + bs]))
        out.append(o.float().cpu())
    torch.cuda.synchronize(); return torch.cat(out).numpy()
sig = lambda z: 1 / (1 + np.exp(-z))
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def groups(p): return list(pd.Series(np.arange(len(p))).groupby(np.asarray(p)).apply(np.array).values)
GRID = np.linspace(0.02, 0.95, 94)
def best_thr(y, p):
    pr = p[:, None] >= GRID[None, :]; y1 = (y == 1)[:, None]; tp = (pr & y1).sum(0); fp = pr.sum(0) - tp; fn = y1.sum() - tp
    return GRID[np.argmax(2 * tp / np.maximum(2 * tp + fp + fn, 1))]
def ece_of(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); e = 0.0
    for k in range(nb):
        mk = bi == k
        if mk.any(): e += mk.mean() * abs(p[mk].mean() - y[mk].mean())
    return e
print("device:", dev, "| new checkpoints:", NW, "| results:", RS)

In [ ]:
# STEP 2 - load the three sites (A = NIH, B = CheXpert, C = VinBigData)
nih = pd.read_csv(r"G:\nih_split.csv"); off = nih.official.values == "test"; te_i = np.where(off)[0]
tr_i = np.where(nih.split_rev.values == "train")[0]; va_i = np.where(nih.split_rev.values == "val")[0]
NIH_A = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224)); Yn = nih[LABELS6].values.astype(int)
cx = pd.read_csv(r"G:\chexpert_final_clean.csv"); cxt = cx[cx.split == "test"].reset_index(drop=True); cxm = cx[cx.split.isin(["train", "val"])].reset_index(drop=True)
cx_tr = np.where(cxm.split.values == "train")[0]; cx_va = np.where(cxm.split.values == "val")[0]
CX_A = np.memmap(r"G:\cx_trainval_224.u8", dtype=np.uint8, mode="r", shape=(len(cxm), 224, 224)); CX_T = np.load(r"G:\cx_test_224.npy")
Ycm = cxm[LABELS6].values.astype(int); Yc = cxt[LABELS6].values.astype(int)
vb = pd.read_csv(rf"{VW}\vinbig_image_labels.csv"); NVB = len(vb)                      # written by Step 3 of your A+B -> C notebook
VB_A = np.memmap(rf"{VW}\vb_train_224.u8", dtype=np.uint8, mode="r", shape=(NVB, 224, 224))
bad = json.load(open(rf"{VW}\vb_train_progress.json"))["bad"]; assert len(bad) == 0, f"{len(bad)} unreadable VinBigData images: cache order would not match the labels"
Yvb = np.zeros((NVB, L), int)
for f in FIND5: Yvb[:, LABELS6.index(f)] = vb[f].values                                # the Edema column stays 0 and is MASKED everywhere (it does not exist in VinBigData)
VALID_C = np.array([l != "Edema" for l in LABELS6])
print(f"NIH train/val/test {len(tr_i)}/{len(va_i)}/{len(te_i)} | CheXpert train/val/test {len(cx_tr)}/{len(cx_va)}/{len(CX_T)} | VinBigData {NVB}")

In [ ]:
# STEP 3 - VinBigData split for the runs where it is a TRAINING source (image-level split, fixed seed). Your A+B -> C results used all 15,000 images and stay valid.
rng = np.random.default_rng(0); perm = rng.permutation(NVB); vb_va = np.sort(perm[:NVB // 5]); vb_tr = np.sort(perm[NVB // 5:])      # 20% validation: 3,000 of your 15,000 images
pd.DataFrame({"image_id": vb.image_id, "split": np.where(np.isin(np.arange(NVB), vb_va), "val", "train")}).to_csv(rf"{RS}\vinbig_split.csv", index=False)
SRC = {"A": dict(arr=NIH_A, Y=Yn,  tr=tr_i,  va=va_i,  valid=np.ones(L, bool)),
       "B": dict(arr=CX_A,  Y=Ycm, tr=cx_tr, va=cx_va, valid=np.ones(L, bool)),
       "C": dict(arr=VB_A,  Y=Yvb, tr=vb_tr, va=vb_va, valid=VALID_C)}
display(pd.DataFrame({k: {"train images": len(v["tr"]), "validation images": len(v["va"])} for k, v in SRC.items()}).T)
display(pd.DataFrame({"VinBigData train positives": Yvb[vb_tr][:, IDX5].sum(0), "VinBigData validation positives": Yvb[vb_va][:, IDX5].sum(0)}, index=FIND5))

In [ ]:
# STEP 4 - training framework: any one or two sources, masked loss (VinBigData has no Edema), source-only model selection, resumable after every epoch
TAG = {"ERM (natural)": "erm_nat", "ERM (balanced)": "erm_bal", "DANN": "dann", "CORAL": "coral"}
MODE = {"ERM (natural)": "natural", "ERM (balanced)": "balanced", "DANN": "balanced", "CORAL": "balanced"}   # same sampling as your A+B runs
OLD = {("DenseNet", "A"): rf"{OUT}\nih_train\densenet121_nih_e{E_OLD}_seed{{s}}.pt", ("DenseNet", "B"): rf"{OUT}\densenet121_e{E_OLD}_seed{{s}}.pt",
       ("ConvNeXt", "A"): rf"{XD}\convnext_tiny_nih_e{E_OLD}_seed{{s}}.pt",           ("ConvNeXt", "B"): rf"{XD}\convnext_tiny_cx_e{E_OLD}_seed{{s}}.pt"}
def is_old(arch, ts, meth): return (meth == "ERM (natural)" and (arch, ts) in OLD) or (arch == "DenseNet" and ts == "A+B")
def ckpt(arch, ts, meth, s):
    if meth == "ERM (natural)" and (arch, ts) in OLD: return OLD[(arch, ts)].format(s=s)
    if arch == "DenseNet" and ts == "A+B": return rf"{DG}\{TAG[meth]}_e{E_OLD}_seed{s}.pt"
    return rf"{NW}\{arch.lower()}_{ts.replace('+', 'p')}_{TAG[meth]}_e{EPOCHS}_seed{s}.pt"
def ready(arch, ts, meth, s):
    p = ckpt(arch, ts, meth, s)
    if not os.path.exists(p): return False
    if is_old(arch, ts, meth): return True
    h = p.replace(".pt", "_history.csv"); return os.path.exists(h) and len(pd.read_csv(h)) >= EPOCHS

def sample_batches(srcs, mode, seed, ep):
    """yields {source: sorted image indices} for every step. natural: random pooled batches (domain mix follows dataset sizes). balanced: the same number of images per source"""
    rng = np.random.default_rng(seed * 100 + ep); sizes = [len(SRC[k]["tr"]) for k in srcs]; spe = math.ceil(sum(sizes) / BS)
    if mode == "natural" or len(srcs) == 1:
        dom = np.concatenate([np.full(n, i) for i, n in enumerate(sizes)]); pos = np.concatenate([SRC[k]["tr"] for k in srcs]); perm = rng.permutation(len(pos))
        for k in range(spe):
            s = perm[k * BS:(k + 1) * BS]
            yield {srcs[i]: np.sort(pos[s][dom[s] == i]) for i in range(len(srcs)) if (dom[s] == i).any()}
    else:
        h = BS // len(srcs)
        for k in range(spe): yield {s_: np.sort(rng.choice(SRC[s_]["tr"], h, replace=False)) for s_ in srcs}
def make_batch(batch, srcs):
    xs, ys, ms, ds = [], [], [], []
    for k, idx in batch.items():
        xs.append(gpu_batch(SRC[k]["arr"], idx, flip=True)); ys.append(SRC[k]["Y"][idx]); ms.append(np.tile(SRC[k]["valid"], (len(idx), 1))); ds.append(np.full(len(idx), srcs.index(k)))
    t = lambda a, dt: torch.from_numpy(np.concatenate(a)).to(dev).to(dt)
    return torch.cat(xs), t(ys, torch.float32), t(ms, torch.float32), t(ds, torch.long)
def val_scores(m, srcs):                                              # mean AUROC over the findings that exist in each SOURCE validation set (never the target)
    out = {}
    for k in srcs:
        s_ = SRC[k]; P = predict_logits_idx(m, s_["arr"], s_["va"]); Yv = s_["Y"][s_["va"]]
        out[k] = float(np.mean([fauc(Yv[:, j], P[:, j]) for j in range(L) if s_["valid"][j] and 5 <= Yv[:, j].sum() <= len(Yv) - 5]))
    return out

class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lam * g, None
def coral_loss(fa, fb):
    d = fa.size(1); return ((torch.cov(fa.T) - torch.cov(fb.T)) ** 2).sum() / (4 * d * d)

def train_run(arch, ts, meth, seed):
    srcs = ts.split("+"); path = ckpt(arch, ts, meth, seed); last, hist_f = path.replace(".pt", "_last.pt"), path.replace(".pt", "_history.csv")
    if ready(arch, ts, meth, seed): print("skip (finished):", os.path.basename(path)); return
    assert not is_old(arch, ts, meth), "this model already exists from your earlier work"
    assert meth.startswith("ERM") or (len(srcs) == 2 and arch == "DenseNet"), "DANN / CORAL need two sources and are implemented for DenseNet only"
    torch.manual_seed(seed); np.random.seed(seed)
    m = timm.create_model(TIMM[arch], pretrained=True, num_classes=L).to(dev)
    disc = nn.Sequential(nn.Linear(m.num_features, 256), nn.ReLU(), nn.Linear(256, 2)).to(dev) if meth == "DANN" else None
    opt = torch.optim.AdamW(list(m.parameters()) + (list(disc.parameters()) if disc else []), lr=LR, weight_decay=1e-4)
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * spe)
    scaler = torch.amp.GradScaler("cuda"); hist, best, start = [], 0.0, 0
    if os.path.exists(last):
        ck = torch.load(last, map_location=dev, weights_only=False)
        m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
        if disc: disc.load_state_dict(ck["disc"])
        print("resumed, next epoch", start + 1)
    for ep in range(start, EPOCHS):
        m.train(); t0 = time.time(); tl = torch.zeros((), device=dev)
        for k, batch in enumerate(tqdm(sample_batches(srcs, MODE[meth], seed, ep), total=spe, leave=False, desc=f"{arch} {ts} {meth} seed {seed} epoch {ep + 1}/{EPOCHS}")):
            x, y, msk, dm = make_batch(batch, srcs)
            with torch.autocast("cuda"):
                if meth in ("DANN", "CORAL"): f = m.forward_head(m.forward_features(x), pre_logits=True); z = m.get_classifier()(f)
                else: z = m(x)
            ls = (F.binary_cross_entropy_with_logits(z.float(), y, reduction="none") * msk).sum(1) / msk.sum(1)     # mean over the outputs that exist for that image's source
            loss = ls.mean()
            if meth == "CORAL": loss = loss + coral_loss(f.float()[dm == 0], f.float()[dm == 1])
            if meth == "DANN":
                p_ = (ep * spe + k) / (EPOCHS * spe); lam = 2 / (1 + math.exp(-10 * p_)) - 1
                loss = loss + 0.1 * F.cross_entropy(disc(GRL.apply(f.float(), lam)), dm)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.detach()
        sc = val_scores(m, srcs); sel = float(np.mean(list(sc.values())))
        hist.append(dict(epoch=ep + 1, train_loss=float(tl) / spe, **{f"val_{k}": v for k, v in sc.items()}, val_sel=sel, minutes=(time.time() - t0) / 60)); pd.DataFrame(hist).to_csv(hist_f, index=False)
        print(f"{arch} {ts} {meth} seed {seed} epoch {ep + 1} | loss {hist[-1]['train_loss']:.4f} | val AUROC " + " ".join(f"{k} {v:.4f}" for k, v in sc.items()) + f" | {hist[-1]['minutes']:.1f} min")
        if sel > best: best = sel; torch.save(m.state_dict(), path)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sch.state_dict(), scaler=scaler.state_dict(), hist=hist, best=best, ep=ep, disc=disc.state_dict() if disc else None), last)
    if os.path.exists(last): os.remove(last)
    del m, opt, sch, scaler, disc; gc.collect(); torch.cuda.empty_cache()

def run_plan(plan):
    for arch, ts, meth in plan:
        for s in SEEDS:
            try: train_run(arch, ts, meth, s)
            except Exception as e:
                print(f"\nFAILED on {arch} {ts} {meth} seed {s}: {type(e).__name__}: {str(e)[:160]}")
                if "illegal memory" in str(e) or "CUDA error" in str(e): print("CUDA context is broken: Kernel > Restart, rerun Steps 0 to 4, then run the plan again (finished runs are skipped, unfinished ones resume).")
                raise
print("framework ready")

In [ ]:
# STEP 5 (updated) - trains everything that is still missing. Finished runs are skipped, unfinished ones resume.
PLAN_C   = [(a, "C", "ERM (natural)") for a in ARCHS]                                                  # VinBigData-only models
PLAN_DN  = [("DenseNet", ts, "ERM (natural)") for ts in ["B+C", "A+C"]]                                # multi-source DenseNet
PLAN_CX  = [("ConvNeXt", ts, "ERM (natural)") for ts in ["A+B", "B+C", "A+C"]]                         # same-architecture counterparts
PLAN_EXT = [("DenseNet", ts, m_) for m_ in ["ERM (balanced)", "DANN", "CORAL"] for ts in ["B+C", "A+C"]]   # the extra methods (these were never started)
ALL = PLAN_C + PLAN_DN + PLAN_CX + PLAN_EXT
METHODS_TO_TRAIN = ["ERM (natural)", "ERM (balanced)", "DANN", "CORAL"]       # remove a name here to skip that method for now
ALL = [r for r in ALL if r[2] in METHODS_TO_TRAIN]
status = pd.DataFrame([dict(model=f"{a} | {ts} | {m_}", seeds_finished=f"{sum(ready(a, ts, m_, s) for s in SEEDS)}/{len(SEEDS)}") for a, ts, m_ in ALL]); display(status)
TODO = [r for r in ALL if not all(ready(r[0], r[1], r[2], s) for s in SEEDS)]
print(f"{len(TODO)} model sets still to train:", [f"{a} {ts} {m_}" for a, ts, m_ in TODO])
if TODO:
    assert gpu_health(), "GPU not healthy: restart the kernel and rerun Steps 0 to 4 first"
    run_plan(TODO)
print("training plan finished")

In [ ]:
# STEP 6 - predictions of every available model on all three test sites (A = NIH official test, B = CheXpert test, C = all 15,000 VinBigData images)
assert gpu_health(), "GPU not healthy: restart the kernel and rerun Steps 0 to 4 first"
TS_ALL = ["A", "B", "C", "A+B", "B+C", "A+C"]
MODELS = [(a, ts, "ERM (natural)") for a in ARCHS for ts in TS_ALL] + [("DenseNet", ts, m_) for m_ in ["ERM (balanced)", "DANN", "CORAL"] for ts in ["A+B", "B+C", "A+C"]]
TG = {"A": dict(name="NIH",        A=NIH_A, idx=te_i,              Y=Yn[te_i], groups=groups(nih.patient_id.values[off])),
      "B": dict(name="CheXpert",   A=CX_T,  idx=np.arange(len(CX_T)), Y=Yc,      groups=groups(cxt.patient_id.values)),
      "C": dict(name="VinBigData", A=VB_A,  idx=np.arange(NVB),    Y=Yvb,      groups=None)}
lpath = lambda a, ts, m_, s: rf"{RS}\logits_{a}_{ts.replace('+', 'p')}_{TAG[m_]}_seed{s}.npz"
def load_model(arch, ts, meth, s):
    m = timm.create_model(TIMM[arch], pretrained=False, num_classes=L).to(dev)
    m.load_state_dict(torch.load(ckpt(arch, ts, meth, s), map_location=dev, weights_only=True)); return m.eval()
for (a, ts, m_) in MODELS:
    for s in SEEDS:
        if os.path.exists(lpath(a, ts, m_, s)): continue
        if not ready(a, ts, m_, s): print("not trained yet, skipped:", a, ts, m_, "seed", s); continue
        try:
            m = load_model(a, ts, m_, s); z = {T: predict_logits_idx(m, g["A"], g["idx"]) for T, g in TG.items()}
            assert all(np.isfinite(v).all() for v in z.values()), "non-finite logits"
            np.savez(lpath(a, ts, m_, s), **z); del m; gc.collect(); torch.cuda.empty_cache(); print("predicted:", a, ts, m_, "seed", s)
        except Exception as e:
            print(f"\nFAILED on {a} {ts} {m_} seed {s}: {type(e).__name__}: {str(e)[:160]}")
            if "illegal memory" in str(e) or "CUDA error" in str(e): print("CUDA context is broken: Kernel > Restart, rerun Steps 0 to 4 and 6 (finished models are skipped).")
            raise
LGT = {T: {} for T in TG}; AVAIL = set()
for (a, ts, m_) in MODELS:
    if all(os.path.exists(lpath(a, ts, m_, s)) for s in SEEDS):
        AVAIL.add((a, ts, m_))
        for s in SEEDS:
            z = np.load(lpath(a, ts, m_, s))
            for T in TG: LGT[T].setdefault((a, ts, m_), {})[s] = z[T]
print("models with all 3 seeds ready:", len(AVAIL), "of", len(MODELS)); print("missing:", sorted(set(MODELS) - AVAIL))

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)
# STEP 7 - the families to compare on each target, their metrics and a PAIRED bootstrap (patients for NIH and CheXpert, images for VinBigData)
SRCS = {"A": ("B", "C"), "B": ("A", "C"), "C": ("A", "B")}      # sources of the models evaluated on each target
POOL = {"A": "B+C", "B": "A+C", "C": "A+B"}                      # the pooled training set that matches those sources
ME = "ERM (natural)"
def member(T, arch, ts, s, meth=ME):
    z = LGT[T][(arch, ts, meth)][s % len(SEEDS)]; v = VALID_C if ts == "C" else np.ones(L, bool); return z, np.broadcast_to(v, z.shape)
M_ = lambda X, ts, o=0, m=ME: (X, ts, o, m)                      # one ensemble member: architecture, training data, seed offset, training method
def ens(ms):                                                       # average of logits; a model that has no output for a finding (VinBigData-only, Edema) is left out of that finding
    Z = np.stack([z for z, _ in ms]); M = np.stack([v for _, v in ms]).astype(float); return (Z * M).sum(0) / np.maximum(M.sum(0), 1), M.sum(0)[0] > 0
def fam_defs(T):
    s1, s2 = SRCS[T]; P = POOL[T]; D, CAT = {}, {}
    def add(name, cat, members): D[name] = members; CAT[name] = cat
    for X in ARCHS:
        add(f"{X}: {s1}-only", "single source", [M_(X, s1)]); add(f"{X}: {s2}-only", "single source", [M_(X, s2)])
        add(f"{X}: pooled {P}", "pooled training", [M_(X, P)]); add(f"{X}: pooled {P} x2 seeds", "pooled ensemble (control)", [M_(X, P), M_(X, P, 1)])
        add(f"{X}: ensemble {s1} + {s2}", "cross-source ensemble", [M_(X, s1), M_(X, s2)])
        add(f"{X}: ensemble {s1} + {s1}", "same-source ensemble (control)", [M_(X, s1), M_(X, s1, 1)]); add(f"{X}: ensemble {s2} + {s2}", "same-source ensemble (control)", [M_(X, s2), M_(X, s2, 1)])
    for m_ in ["ERM (balanced)", "DANN", "CORAL"]: add(f"DenseNet: pooled {P} [{m_}]", "pooled training (other method)", [M_("DenseNet", P, 0, m_)])
    for s_ in (s1, s2): add(f"{s_}-only: DenseNet + ConvNeXt", "cross-architecture ensemble (control)", [M_("DenseNet", s_), M_("ConvNeXt", s_)])
    add(f"4 models: {s1}-only + {s2}-only, both architectures", "cross-source + cross-architecture", [M_(X, s_) for X in ARCHS for s_ in (s1, s2)])
    add(f"pooled {P}: DenseNet + ConvNeXt", "pooled ensemble (control)", [M_(X, P) for X in ARCHS])
    add(f"pooled {P} x4 (2 archs x 2 seeds)", "pooled ensemble (control)", [M_(X, P, o) for X in ARCHS for o in (0, 1)])
    keep = {n: mm for n, mm in D.items() if all((x, ts_, m_) in AVAIL for x, ts_, _, m_ in mm)}
    return keep, {n: CAT[n] for n in keep}
def contrasts(T, F_):
    s1, s2 = SRCS[T]; P = POOL[T]; C = []
    for X in ARCHS:
        C += [(f"{X}: ensemble {s1} + {s2}", f"{X}: pooled {P}", "cross-source ensemble vs pooled training"),
              (f"{X}: ensemble {s1} + {s2}", f"{X}: pooled {P} x2 seeds", "cross-source ensemble vs pooled x2 (same inference cost)"),
              (f"{X}: ensemble {s1} + {s2}", f"{X}: ensemble {s1} + {s1}", f"cross-source vs same-source ensemble ({s1})"),
              (f"{X}: ensemble {s1} + {s2}", f"{X}: ensemble {s2} + {s2}", f"cross-source vs same-source ensemble ({s2})"),
              (f"{X}: pooled {P}", f"{X}: {s1}-only", f"pooled training vs {s1}-only"), (f"{X}: pooled {P}", f"{X}: {s2}-only", f"pooled training vs {s2}-only")]
    for m_ in ["ERM (balanced)", "DANN", "CORAL"]:
        C += [(f"DenseNet: pooled {P} [{m_}]", f"DenseNet: pooled {P}", f"pooled [{m_}] vs pooled ERM (natural)"), (f"DenseNet: ensemble {s1} + {s2}", f"DenseNet: pooled {P} [{m_}]", f"cross-source ensemble vs pooled [{m_}]")]
    f4 = f"4 models: {s1}-only + {s2}-only, both architectures"
    C += [(f4, f"pooled {P} x4 (2 archs x 2 seeds)", "4-model cross-source vs pooled x4"), (f4, f"{s1}-only: DenseNet + ConvNeXt", f"4-model vs cross-architecture only ({s1})"),
          (f4, f"{s2}-only: DenseNet + ConvNeXt", f"4-model vs cross-architecture only ({s2})"), (f4, f"pooled {P}: DenseNet + ConvNeXt", "4-model vs pooled DenseNet + ConvNeXt"),
          (f"ConvNeXt: pooled {P}", f"DenseNet: pooled {P}", "architecture effect (pooled)")]
    return [c for c in C if c[0] in F_ and c[1] in F_]

B = 300                                                           # bootstrap resamples (use 200 to save time)
FD, FCAT, FL, PTA, BSA, MET, CON = {}, {}, {}, {}, {}, {}, {}
for T, g in TG.items():
    FD[T], FCAT[T] = fam_defs(T); Y = g["Y"]; cols = list(range(L)) if T != "C" else IDX5
    FL[T] = {n: {s: ens([member(T, X, ts, s + o, m_) for X, ts, o, m_ in mm])[0] for s in SEEDS} for n, mm in FD[T].items()}      # logits of every family, every seed
    vc = {n: np.logical_or.reduce([ens([member(T, X, ts, 0, m_)])[1] for X, ts, _, m_ in mm]) for n, mm in FD[T].items()}            # findings each family can predict
    PTA[T] = {n: np.array([np.mean([fauc(Y[:, j], FL[T][n][s][:, j]) for s in SEEDS]) if (j in cols and vc[n][j]) else np.nan for j in range(L)]) for n in FD[T]}
    rng = np.random.default_rng(0); BSA[T] = {n: np.full((B, L), np.nan) for n in FD[T]}
    for b in tqdm(range(B), desc=f"bootstrap, target {T} ({g['name']})"):
        gr = g["groups"]; ix = np.concatenate([gr[k] for k in rng.integers(0, len(gr), len(gr))]) if gr is not None else rng.integers(0, len(Y), len(Y))
        for n in FD[T]:
            for j in cols:
                if vc[n][j]: BSA[T][n][b, j] = np.mean([fauc(Y[ix, j], FL[T][n][s][ix, j]) for s in SEEDS])
    rows = []
    for n in FD[T]:
        P_s = [sig(FL[T][n][s]) for s in SEEDS]
        ap = [np.mean([average_precision_score(Y[:, j], p[:, j]) for j in IDX5]) for p in P_s]; ec = [np.mean([ece_of(Y[:, j], p[:, j]) for j in IDX5]) for p in P_s]
        m5, lo5, hi5 = np.nanmean(PTA[T][n][IDX5]), *np.percentile(np.nanmean(BSA[T][n][:, IDX5], 1), [2.5, 97.5])
        rows.append({"model": n, "category": FCAT[T][n], "AUROC (5 findings)": m5, "95% CI": f"[{lo5:.3f}, {hi5:.3f}]",
                     "AUROC (6 findings)": np.nanmean(PTA[T][n]) if (T != "C" and not np.isnan(PTA[T][n]).any()) else np.nan, "AUPRC": np.mean(ap), "ECE": np.mean(ec)})
    MET[T] = pd.DataFrame(rows).set_index("model")
    rows = []
    for a, b_, lab_ in contrasts(T, FD[T]):
        d = np.nanmean(BSA[T][a][:, IDX5], 1) - np.nanmean(BSA[T][b_][:, IDX5], 1); lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"A": a, "minus B": b_, "comparison": lab_, "mean diff": round(float(d.mean()), 4), "95% CI": f"[{lo:+.4f}, {hi:+.4f}]", "excludes 0": "yes" if (lo > 0 or hi < 0) else "no", "lo": lo, "hi": hi})
    CON[T] = pd.DataFrame(rows)
print("done")

In [ ]:
# STEP 8 - tables
for T, g in TG.items():
    print(f"\n===== target {T}: {g['name']} (models trained on {SRCS[T][0]} and {SRCS[T][1]}) ====="); display(MET[T].round(3)); display(CON[T].drop(columns=["lo", "hi"]))
    MET[T].to_csv(rf"{RS}\table_{T}_{g['name']}_models.csv"); CON[T].to_csv(rf"{RS}\table_{T}_{g['name']}_contrasts.csv", index=False)
rows = []
for T in TG:
    for n in FD[T]:
        for j, f in enumerate(LABELS6):
            if not np.isnan(PTA[T][n][j]): rows.append(dict(target=TG[T]["name"], model=n, finding=f, auroc=PTA[T][n][j], lo=np.nanpercentile(BSA[T][n][:, j], 2.5), hi=np.nanpercentile(BSA[T][n][:, j], 97.5)))
pd.DataFrame(rows).to_csv(rf"{RS}\table_per_finding_auroc_with_CI.csv", index=False)

# full transfer matrix (single models and pooled models, 5 common findings; cells whose training data contains the test site are marked)
def auc5(arch, ts, T):
    if (arch, ts, ME) not in AVAIL: return np.nan
    Y = TG[T]["Y"]; return np.mean([np.mean([fauc(Y[:, j], LGT[T][(arch, ts, ME)][s][:, j]) for j in IDX5]) for s in SEEDS])
MAT = {a: pd.DataFrame({TG[T]["name"]: {ts: (auc5(a, ts, T) if not (T == "C" and "C" in ts) else np.nan) for ts in TS_ALL} for T in TG}) for a in ARCHS}
for a in ARCHS: print(f"\n{a}: rows = training data, columns = test site (mean AUROC, 5 common findings; n/a = VinBigData was used for training)"); display(MAT[a].round(3)); MAT[a].to_csv(rf"{RS}\table_transfer_matrix_{a}.csv")

In [ ]:
# STEP 9 - figures
CATORD = ["single source", "pooled training", "pooled training (other method)", "pooled ensemble (control)", "same-source ensemble (control)", "cross-architecture ensemble (control)", "cross-source ensemble", "cross-source + cross-architecture"]
CATCOL = {"single source": "0.85", "pooled training": "0.3", "pooled training (other method)": "0.45", "pooled ensemble (control)": "0.55", "same-source ensemble (control)": "0.7", "cross-architecture ensemble (control)": "C0",
          "cross-source ensemble": "C3", "cross-source + cross-architecture": "C1"}
order = lambda T: sorted(FD[T], key=lambda n: (CATORD.index(FCAT[T][n]), n))
m5ci = lambda T, n: (np.nanmean(PTA[T][n][IDX5]), *np.percentile(np.nanmean(BSA[T][n][:, IDX5], 1), [2.5, 97.5]))
def cat_legend(fig, cats, ncol=3):
    from matplotlib.patches import Patch
    cats = sorted(set(cats), key=CATORD.index); fig.legend(handles=[Patch(facecolor=CATCOL[c], edgecolor="k", lw=0.4, label=c) for c in cats], frameon=False, fontsize=5.5, loc="lower center", ncol=ncol)
def ylims(T, names, low=0.04):
    v = [m5ci(T, n) for n in names]; return min(x[1] for x in v) - low, max(x[2] for x in v) + 0.012

# 9A: full transfer matrix (training data x test site)
fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.0)); allv = np.concatenate([MAT[a].values.ravel() for a in ARCHS]); lo_, hi_ = np.nanmin(allv) - 0.02, np.nanmax(allv) + 0.01
for a_, a in zip(ax, ARCHS):
    D = MAT[a].values; a_.imshow(np.ma.masked_invalid(D), cmap="viridis", vmin=lo_, vmax=hi_, aspect="auto")
    for i in range(D.shape[0]):
        for j, T in enumerate(TG):
            seen = T in TS_ALL[i].split("+"); txt = "n/a" if np.isnan(D[i, j]) else f"{D[i, j]:.3f}" + ("†" if seen else "")
            a_.text(j, i, txt, ha="center", va="center", fontsize=6.5, color="k" if np.isnan(D[i, j]) or D[i, j] > (lo_ + hi_) / 2 else "w")
    a_.set_xticks(range(3)); a_.set_xticklabels([TG[T]["name"] for T in TG], rotation=20, ha="right"); a_.set_yticks(range(len(TS_ALL))); a_.set_yticklabels(TS_ALL); a_.set_title(a); a_.set_xlabel("Test site"); a_.grid(False)
ax[0].set_ylabel("Training data"); fig.text(0.5, -0.02, "† the test site was part of the training data;  n/a: VinBigData was used for training", ha="center", fontsize=6); fig.tight_layout(); save_fig(fig, "fig_transfer_matrix")

# 9B: AUROC of every family on every target (95% bootstrap CI)
for T in TG:
    names = order(T); fig, ax = plt.subplots(figsize=(COL2, 4.0))
    for i, n in enumerate(names):
        m5, lo, hi = m5ci(T, n); ax.bar(i, m5, yerr=[[m5 - lo], [hi - m5]], capsize=1.5, color=CATCOL[FCAT[T][n]], edgecolor="k", lw=0.4)
    ax.set_ylim(*ylims(T, names, 0.05)); ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=60, ha="right", fontsize=5.5)
    ax.set_ylabel("AUROC (mean of 5 findings)"); ax.set_title(f"Test site: {TG[T]['name']} (models trained on {SRCS[T][0]} and {SRCS[T][1]})"); ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout(rect=[0, 0.07, 1, 1]); cat_legend(fig, [FCAT[T][n] for n in names], 4); save_fig(fig, f"fig_auroc_{T}_{TG[T]['name']}")

# 9C: the paired comparisons that matter (difference in AUROC with 95% CI; red = CI excludes 0)
PRIM = ("cross-source ensemble vs pooled training", "cross-source ensemble vs pooled x2", "cross-source ensemble vs pooled [", "cross-source vs same-source", "pooled training vs", "pooled [", "4-model cross-source vs pooled x4")
fig, axs = plt.subplots(3, 1, figsize=(COL2, 8.6))
for a_, T in zip(axs, TG):
    d = CON[T][CON[T].comparison.str.startswith(PRIM)].reset_index(drop=True)
    for i, r in d.iterrows(): a_.plot([r.lo, r.hi], [i, i], color="k", lw=1); a_.plot(r["mean diff"], i, "o", color="C3" if r["excludes 0"] == "yes" else "0.5", ms=3.5)
    a_.axvline(0, color="k", lw=0.6, ls=":"); a_.set_yticks(range(len(d))); a_.invert_yaxis()
    a_.set_yticklabels([f"{r.A.split(':')[0] if r.A.split(':')[0] in ARCHS else 'both archs'}: {r.comparison}" for _, r in d.iterrows()], fontsize=5.5)
    a_.set_title(f"Test site: {TG[T]['name']}"); a_.set_xlabel("Difference in AUROC (first model minus second)"); a_.grid(True, axis="x", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_paired_contrasts")

# 9D: what does adding a second source add? (pooled model minus each of its single-source models, per finding)
fig, axs = plt.subplots(3, 2, figsize=(COL2, 7.4))
for r_, T in enumerate(TG):
    s1, s2 = SRCS[T]; P = POOL[T]
    for c_, X in enumerate(ARCHS):
        a_ = axs[r_, c_]; pn = f"{X}: pooled {P}"
        for k, (sx, col) in enumerate([(s1, "0.3"), (s2, "0.75")]):
            sn = f"{X}: {sx}-only"
            if pn not in BSA[T] or sn not in BSA[T]: continue
            dd = BSA[T][pn] - BSA[T][sn]; mu, lo, hi = np.nanmean(dd, 0), *np.nanpercentile(dd, [2.5, 97.5], 0); m5 = np.nanmean(dd[:, IDX5], 1)
            mu, lo, hi = np.append(mu, m5.mean()), np.append(lo, np.percentile(m5, 2.5)), np.append(hi, np.percentile(m5, 97.5))
            a_.bar(np.arange(L + 1) + (k - 0.5) * 0.38, mu, 0.38, yerr=[np.clip(mu - lo, 0, None), np.clip(hi - mu, 0, None)], capsize=1.2, color=col, edgecolor="k", lw=0.4, label=f"pooled {P} minus {sx}-only")
        a_.axhline(0, color="k", lw=0.6); a_.set_xticks(range(L + 1)); a_.set_xticklabels(LABELS6 + ["mean (5)"], rotation=40, ha="right", fontsize=5.5)
        a_.set_title(f"{X}, test site {TG[T]['name']}", fontsize=7); a_.set_ylabel("Difference in AUROC"); a_.legend(frameon=True, framealpha=0.85, edgecolor="none", fontsize=5.5); a_.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_adding_a_source")

# 9E: the fair ensemble comparison (same architecture) and the cross-architecture comparison
fig, axs = plt.subplots(3, 1, figsize=(COL2, 8.6))
for a_, T in zip(axs, TG):
    s1, s2 = SRCS[T]; P = POOL[T]; names = []
    for X in ARCHS: names += [f"{X}: {s1}-only", f"{X}: {s2}-only", f"{X}: ensemble {s1} + {s1}", f"{X}: ensemble {s2} + {s2}", f"{X}: pooled {P}", f"{X}: pooled {P} x2 seeds", f"{X}: ensemble {s1} + {s2}"]
    names = [n for n in names if n in FD[T]]; xs = [i + (0.8 if n.startswith("ConvNeXt") else 0) for i, n in enumerate(names)]
    for x_, n in zip(xs, names): m5, lo, hi = m5ci(T, n); a_.bar(x_, m5, yerr=[[m5 - lo], [hi - m5]], capsize=1.5, color=CATCOL[FCAT[T][n]], edgecolor="k", lw=0.4)
    a_.set_ylim(*ylims(T, names)); a_.set_xticks(xs); a_.set_xticklabels([n.replace("DenseNet: ", "D: ").replace("ConvNeXt: ", "C: ") for n in names], rotation=45, ha="right", fontsize=5.5)
    a_.set_title(f"Test site: {TG[T]['name']} (D = DenseNet, C = ConvNeXt)"); a_.set_ylabel("AUROC (5 findings)"); a_.grid(True, axis="y", alpha=0.3)
fig.tight_layout(rect=[0, 0.04, 1, 1]); cat_legend(fig, [FCAT[T][n] for T in TG for n in FD[T] if n.split(":")[0] in ARCHS and "[" not in n], 3); save_fig(fig, "fig_fair_ensembles_same_architecture")
fig, axs = plt.subplots(1, 3, figsize=(COL2, 3.6))
for a_, T in zip(axs, TG):
    s1, s2 = SRCS[T]; P = POOL[T]; names = [n for n in [f"{s1}-only: DenseNet + ConvNeXt", f"{s2}-only: DenseNet + ConvNeXt", f"pooled {P}: DenseNet + ConvNeXt", f"pooled {P} x4 (2 archs x 2 seeds)", f"4 models: {s1}-only + {s2}-only, both architectures"] if n in FD[T]]
    for i, n in enumerate(names): m5, lo, hi = m5ci(T, n); a_.bar(i, m5, yerr=[[m5 - lo], [hi - m5]], capsize=1.5, color=CATCOL[FCAT[T][n]], edgecolor="k", lw=0.4)
    a_.set_ylim(*ylims(T, names, 0.03)); a_.set_xticks(range(len(names))); a_.set_xticklabels(names, rotation=60, ha="right", fontsize=5); a_.set_title(TG[T]["name"]); a_.grid(True, axis="y", alpha=0.3)
axs[0].set_ylabel("AUROC (5 findings)"); fig.tight_layout(rect=[0, 0.07, 1, 1]); cat_legend(fig, ["cross-architecture ensemble (control)", "pooled ensemble (control)", "cross-source + cross-architecture"], 3); save_fig(fig, "fig_fair_ensembles_cross_architecture")

# 9F: per-finding AUROC of every family
for T in TG:
    names = order(T); H = np.array([PTA[T][n] for n in names]); fig, ax = plt.subplots(figsize=(COL2, 0.22 * len(names) + 1.4))
    ax.imshow(np.ma.masked_invalid(H), cmap="viridis", aspect="auto")
    for i in range(H.shape[0]):
        for j in range(L):
            if not np.isnan(H[i, j]): ax.text(j, i, f"{H[i, j]:.3f}", ha="center", va="center", fontsize=5.5, color="w" if H[i, j] < np.nanmean(H) else "k")
    ax.set_xticks(range(L)); ax.set_xticklabels(LABELS6, rotation=30, ha="right"); ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=5.5); ax.set_title(f"AUROC per finding, test site {TG[T]['name']}"); ax.grid(False)
    fig.tight_layout(); save_fig(fig, f"fig_per_finding_{T}_{TG[T]['name']}")

# 9G: calibration and AUPRC
fig, axs = plt.subplots(2, 3, figsize=(COL2, 6.2))
for c_, T in enumerate(TG):
    names = order(T)
    for r_, (col, ttl) in enumerate([("ECE", "ECE (lower is better)"), ("AUPRC", "AUPRC (higher is better)")]):
        a_ = axs[r_, c_]; a_.bar(range(len(names)), [MET[T].loc[n, col] for n in names], color=[CATCOL[FCAT[T][n]] for n in names], edgecolor="k", lw=0.4)
        a_.set_xticks(range(len(names))); a_.set_xticklabels(names, rotation=90, fontsize=4.5); a_.set_title(f"{TG[T]['name']}: {ttl}", fontsize=6.5); a_.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_calibration_auprc")

In [ ]:
# STEP 10 - Grad-CAM. (a) correct and incorrect cases of the pooled model; (b) the SAME random positive images seen by the single-source model and by the pooled model
# target A (NIH): trained on B only versus B + C | target B (CheXpert): trained on A only versus A + C | seed 0 | random samples, not hand-picked
assert gpu_health(), "GPU not healthy: restart the kernel and rerun Steps 0 to 4 and 6 first"
GARCH, GFIND, GN, GK = ["DenseNet", "ConvNeXt"], ["Cardiomegaly", "Edema", "Pleural Effusion", "Pneumothorax"], 200, 4
GTARG = {"A": ("B", "B+C"), "B": ("A", "A+C")}
SHOWN = {("DenseNet", "A", "Cardiomegaly"), ("DenseNet", "B", "Pleural Effusion")}
GROUPS = ["TP", "FN", "FP", "TN"]; GNAME = {"TP": "Correct positive (TP)", "FN": "Missed positive (FN)", "FP": "False alarm (FP)", "TN": "Correct negative (TN)"}
EDGE = np.ones((224, 224), bool); EDGE[33:-33, 33:-33] = False; BOTTOM = np.zeros((224, 224), bool); BOTTOM[149:, :] = True
def gradcam(model, imgs, cls, bs=25):
    cams = []
    for i in range(0, len(imgs), bs):
        x = (torch.from_numpy(np.ascontiguousarray(imgs[i:i + bs])).to(dev).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1) - MEAN) / STD
        with torch.enable_grad():
            fm = model.forward_features(x); out = model.forward_head(fm)
            assert fm.requires_grad, "feature map has no gradient (check requires_grad)"
            g = torch.autograd.grad(out[:, cls].sum(), fm)[0]
        cam = torch.relu((g.mean((2, 3), keepdim=True) * fm).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[:, 0]
        cams.append((cam / (cam.flatten(1).max(1).values.view(-1, 1, 1) + 1e-8)).detach().cpu().numpy()); del fm, out, g, cam, x
    torch.cuda.empty_cache(); return np.concatenate(cams)
get_img = lambda T, ix: NIH_A[te_i[ix]] if T == "A" else CX_T[ix]            # ix must be sorted
def source_thr(m, srcs):                                                      # max-F1 thresholds from the SOURCE validation sets of the pooled model
    pr = {k: (sig(predict_logits_idx(m, SRC[k]["arr"], SRC[k]["va"])), SRC[k]["Y"][SRC[k]["va"]]) for k in srcs}; thr = np.full(L, 0.5)
    for j in range(L):
        ks = [k for k in srcs if SRC[k]["valid"][j]]; thr[j] = best_thr(np.concatenate([pr[k][1][:, j] for k in ks]), np.concatenate([pr[k][0][:, j] for k in ks]))
    return thr
GST, GBA = [], []
for arch in GARCH:
    for T, (s_ts, p_ts) in GTARG.items():
        if not (ready(arch, s_ts, ME, 0) and ready(arch, p_ts, ME, 0)): print("skipped (models not ready):", arch, T); continue
        m_s, m_p = load_model(arch, s_ts, ME, 0), load_model(arch, p_ts, ME, 0)
        for mm_ in (m_s, m_p):
            for p_ in mm_.parameters(): p_.requires_grad_(True)
        thr = source_thr(m_p, p_ts.split("+")); Y = TG[T]["Y"]; Pp = sig(LGT[T][(arch, p_ts, ME)][0]); rng = np.random.default_rng(11)
        for fnd in GFIND:
            j = LABELS6.index(fnd); y, p, t = Y[:, j], Pp[:, j], thr[j]; base = rf"{RS}\gradcam_{arch}_{T}_{fnd.replace(' ', '_')}"
            ids = {"TP": np.where((y == 1) & (p >= t))[0], "FN": np.where((y == 1) & (p < t))[0], "FP": np.where((y == 0) & (p >= t))[0], "TN": np.where((y == 0) & (p < t))[0]}
            fig, axs = plt.subplots(4, GK, figsize=(5.4, 5.6))                                                                    # (a) correct / incorrect cases of the pooled model
            for r_, g_ in enumerate(GROUPS):
                for c_ in range(GK): axs[r_, c_].axis("off")
                axs[r_, 0].text(-0.04, 0.5, f"{GNAME[g_]}\nn = {len(ids[g_])}", transform=axs[r_, 0].transAxes, rotation=90, va="center", ha="right", fontsize=5.5)
                n = min(GN, len(ids[g_]))
                if n == 0: continue
                pk = np.sort(rng.choice(ids[g_], n, replace=False)); imgs = get_img(T, pk); cams = gradcam(m_p, imgs, j)
                ok = cams.max((1, 2)) > 0.5; s_ = np.maximum(cams.sum((1, 2)), 1e-8)
                GST.append(dict(model=arch, target=TG[T]["name"], trained_on=p_ts, finding=fnd, group=g_, n_group=len(ids[g_]), n_used=n, empty_maps=int((~ok).sum()),
                                edge=(cams[:, EDGE].sum(1) / s_)[ok].mean() if ok.any() else np.nan, bottom=(cams[:, BOTTOM].sum(1) / s_)[ok].mean() if ok.any() else np.nan))
                for c_ in range(min(GK, n)):
                    axs[r_, c_].imshow(imgs[c_], cmap="gray"); axs[r_, c_].imshow(cams[c_], cmap="jet", alpha=0.4); axs[r_, c_].set_title(f"p = {p[pk[c_]]:.2f}", fontsize=5.5, pad=1)
            fig.suptitle(f"{arch} trained on {p_ts}, tested on {TG[T]['name']}: {fnd} [threshold {t:.2f}]", fontsize=7); fig.tight_layout()
            fig.savefig(base + "_cases.pdf", bbox_inches="tight"); fig.savefig(base + "_cases.png", dpi=300, bbox_inches="tight")
            if (arch, T, fnd) in SHOWN: plt.show()
            plt.close(fig)
            pos = np.where(y == 1)[0]; pk = np.sort(rng.choice(pos, min(GN, len(pos)), replace=False)); imgs = get_img(T, pk)         # (b) same positives, two models
            cs, cp = gradcam(m_s, imgs, j), gradcam(m_p, imgs, j); okb = (cs.max((1, 2)) > 0.5) & (cp.max((1, 2)) > 0.5)
            corr = np.array([np.corrcoef(a_.ravel(), b_.ravel())[0, 1] for a_, b_ in zip(cs[okb], cp[okb])]); ss, sp = np.maximum(cs.sum((1, 2)), 1e-8), np.maximum(cp.sum((1, 2)), 1e-8)
            GBA.append(dict(model=arch, target=TG[T]["name"], single=f"{s_ts}-only", pooled=p_ts, finding=fnd, n_positives=len(pos), n_used=int(okb.sum()), map_correlation=np.nanmean(corr) if len(corr) else np.nan,
                            edge_single=(cs[:, EDGE].sum(1) / ss)[okb].mean(), edge_pooled=(cp[:, EDGE].sum(1) / sp)[okb].mean(), bottom_single=(cs[:, BOTTOM].sum(1) / ss)[okb].mean(), bottom_pooled=(cp[:, BOTTOM].sum(1) / sp)[okb].mean()))
            fig, axs = plt.subplots(2, GK, figsize=(5.4, 3.0))
            for c_ in range(GK):
                for r_, cm_ in enumerate((cs, cp)): axs[r_, c_].imshow(imgs[c_], cmap="gray"); axs[r_, c_].imshow(cm_[c_], cmap="jet", alpha=0.4); axs[r_, c_].axis("off")
            axs[0, 0].text(-0.04, 0.5, f"trained on {s_ts} only", transform=axs[0, 0].transAxes, rotation=90, va="center", ha="right", fontsize=6)
            axs[1, 0].text(-0.04, 0.5, f"trained on {p_ts}", transform=axs[1, 0].transAxes, rotation=90, va="center", ha="right", fontsize=6)
            fig.suptitle(f"{arch}, {TG[T]['name']} test, {fnd}: same random positive images (mean map correlation {GBA[-1]['map_correlation']:.2f})", fontsize=7); fig.tight_layout()
            fig.savefig(base + "_before_after.pdf", bbox_inches="tight"); fig.savefig(base + "_before_after.png", dpi=300, bbox_inches="tight")
            if (arch, T, fnd) in SHOWN: plt.show()
            plt.close(fig); print("Grad-CAM done:", arch, TG[T]["name"], fnd)
        del m_s, m_p; gc.collect(); torch.cuda.empty_cache()
GST, GBA = pd.DataFrame(GST), pd.DataFrame(GBA); GST.to_csv(rf"{RS}\gradcam_cases_region_stats.csv", index=False); GBA.to_csv(rf"{RS}\gradcam_before_after.csv", index=False)
print("uniform-heat baselines: edge", round(EDGE.mean(), 2), "| bottom third", round(BOTTOM.mean(), 2)); display(GBA.round(3)); display(GST.round(3))
if len(GBA):
    fig, axs = plt.subplots(1, 2, figsize=(COL2, 3.0)); lab_ = [f"{r.model[:5]}, {r.target}\n{r.finding}" for r in GBA.itertuples()]
    axs[0].bar(range(len(GBA)), GBA.map_correlation, color="0.5", edgecolor="k", lw=0.4); axs[0].set(ylabel="Correlation of the two Grad-CAM maps", ylim=(0, 1)); axs[0].set_title("Does adding a second source change where the model looks?")
    axs[1].bar(np.arange(len(GBA)) - 0.2, GBA.edge_single, 0.4, color="0.8", edgecolor="k", lw=0.4, label="single source"); axs[1].bar(np.arange(len(GBA)) + 0.2, GBA.edge_pooled, 0.4, color="0.3", edgecolor="k", lw=0.4, label="pooled")
    axs[1].axhline(EDGE.mean(), color="k", ls=":", lw=0.6); axs[1].set(ylabel="Fraction of heat in the outer 15% frame"); axs[1].legend(frameon=False)
    for a_ in axs: a_.set_xticks(range(len(GBA))); a_.set_xticklabels(lab_, rotation=90, fontsize=4.5); a_.grid(True, axis="y", alpha=0.3)
    fig.tight_layout(); save_fig(fig, "fig_gradcam_summary")

In [ ]:
# STEP 11 - save everything into one bundle (checkpoints stay where they are)
BUN = rf"{OUT}\FAIR_ENS_bundle"; os.makedirs(BUN, exist_ok=True)
shutil.copytree(RS, rf"{BUN}\fair_ens", dirs_exist_ok=True)
shutil.copytree(NW, rf"{BUN}\training_histories", dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.pt"))
json.dump(dict(date=str(datetime.date.today()), epochs=EPOCHS, seeds=SEEDS, batch_size=BS, lr=LR, sources_and_targets={T: dict(test=TG[T]["name"], trained_on=SRCS[T], pooled=POOL[T]) for T in TG},
               vinbig_split="12000 train / 3000 validation, image-level, seed 0 (only for runs where VinBigData is a source)", edema="masked in the loss for VinBigData images",
               model_selection="mean validation AUROC over the source domains only", bootstrap=B, bootstrap_unit="patients for NIH and CheXpert, images for VinBigData (paired across models)",
               ensembles="average of logits; a VinBigData-only model contributes no Edema output", gradcam=dict(seed=0, images_per_group=GN, findings=GFIND), torch=torch.__version__, timm=timm.__version__),
          open(rf"{BUN}\settings_fair_ens.json", "w"), indent=2)
zp = shutil.make_archive(rf"{OUT}\FAIR_ENS_bundle", "zip", BUN); print("bundle:", zp, f"({os.path.getsize(zp) / 1e6:.0f} MB)")
for e in [".csv", ".pdf", ".png", ".npz"]: print(e, sum(f.endswith(e) for _, _, fs in os.walk(BUN) for f in fs))